In [ ]:
# Initialization, styling configuration, and helper utilities
%matplotlib inline

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from torch_geometric.datasets import KarateClub
    from torch_geometric.nn import GCNConv
    PYG_AVAILABLE = True
except ImportError:
    PYG_AVAILABLE = False

# Load course helper utilities (local file or direct GitHub fetch for standalone Colab)
try:
    import helpers as hp
except ImportError:
    import urllib.request
    _url = "https://raw.githubusercontent.com/konstantin-schekotihin/dl_course/master/ML-DL/02_deep_learning/helpers.py"
    urllib.request.urlretrieve(_url, "helpers.py")
    import helpers as hp

import importlib
importlib.reload(hp)

hp.setup_theme()

# Graph Neural Networks
## Geometric Deep Learning, Permutation Symmetries, and Message Passing

This lecture explores deep learning on relational graph-structured data:
- **Non-Euclidean Topologies**: Nodes $\mathcal{V}$, edges $\mathcal{E}$, adjacency matrix $\mathbf{A}$, node features $\mathbf{X}$
- **Graph Symmetries**: Permutation invariance and equivariance
- **The Message Passing Framework (MPNN)**: Message, Aggregation, and Update functions
- **Graph Convolutional Networks (GCN)**: Spectral and spatial formulations (Kipf & Welling 2017)
- **Graph Attention Networks (GAT)**: Learned edge coefficients (Veličković et al. 2018)
- **Implementation**: Semi-supervised node classification on Zachary's Karate Club in PyG

# Graphs as Non-Euclidean Data
## Relational Topologies

Many real-world domains cannot be represented as regular 1D sequences or 2D image grids:
- **Molecules & Chemistry**: Atoms connected by covalent bonds
- **Social Networks**: Users connected by friendship/follow ties
- **Citation Networks**: Papers connected by bibliographic citations

A graph $\mathcal{G} = (\mathcal{V}, \mathcal{E})$ is formalized by:
- **Node Set**: $\mathcal{V} = \{v_1, \dots, v_N\}$ with feature matrix $\mathbf{X} \in \mathbb{R}^{N \times D}$
- **Edge Set**: $\mathcal{E} \subseteq \mathcal{V} \times \mathcal{V}$ with adjacency matrix $\mathbf{A} \in \{0, 1\}^{N \times N}$:
  $$A_{ij} = \begin{cases} 1, & \text{if } (v_i, v_j) \in \mathcal{E} \\ 0, & \text{otherwise} \end{cases}$$
- **Degree Matrix**: Diagonal matrix $D_{ii} = \sum_j A_{ij}$ counting connections per node.

## Symmetries on Graphs
## Permutation Invariance & Equivariance

Nodes in a graph have no canonical ordering. Any permutation matrix $\mathbf{P} \in \{0, 1\}^{N \times N}$ simply re-indexes the nodes without changing the underlying topology:
$$\mathbf{A}^\prime = \mathbf{P} \mathbf{A} \mathbf{P}^\mathrm{T}, \quad \mathbf{X}^\prime = \mathbf{P} \mathbf{X}$$

A valid Graph Neural Network must respect graph symmetries:
1. **Permutation Equivariance (Node-Level Tasks)**:
   Re-indexing the inputs re-indexes the output representations identically:
   $$f\left(\mathbf{P}\mathbf{A}\mathbf{P}^\mathrm{T}, \mathbf{P}\mathbf{X}\right) = \mathbf{P} f(\mathbf{A}, \mathbf{X})$$
2. **Permutation Invariance (Graph-Level Tasks)**:
   Predicting molecular solubility or graph properties must yield identical scalars regardless of node indexing:
   $$g\left(\mathbf{P}\mathbf{A}\mathbf{P}^\mathrm{T}, \mathbf{P}\mathbf{X}\right) = g(\mathbf{A}, \mathbf{X})$$

Standard MLPs and CNNs fail because their fixed weight indices depend strictly on arbitrary input ordering.

# The Message Passing Neural Network (MPNN)
## Message, Aggregate, and Update

Justin Gilmer et al. (2017) unified graph architectures under the **Message Passing** paradigm:

At layer $l$, each node $v$ updates its representation vector $\mathbf{h}_v^{(l)}$ through three steps:

1. **Message Generation**:
   Compute message along edge $(u, v)$ from neighbor $u \in \mathcal{N}(v)$:
   $$\mathbf{m}_{uv}^{(l)} = M^{(l)}\left(\mathbf{h}_u^{(l)}, \mathbf{h}_v^{(l)}, \mathbf{e}_{uv}\right)$$

2. **Permutation-Invariant Aggregation**:
   Aggregate messages using a symmetric reduction operator (sum, mean, max):
   $$\mathbf{m}_v^{(l)} = \bigoplus_{u \in \mathcal{N}(v)} \mathbf{m}_{uv}^{(l)}$$

3. **Node State Update**:
   Update node hidden state:
   $$\mathbf{h}_v^{(l+1)} = U^{(l)}\left(\mathbf{h}_v^{(l)}, \mathbf{m}_v^{(l)}\right)$$

# Graph Convolutional Networks (GCN)
## Kipf & Welling (2017)

Thomas Kipf and Max Welling formulated **GCN** as a localized first-order spectral approximation:

$$\mathbf{H}^{(l+1)} = \sigma\left(\tilde{\mathbf{D}}^{-\frac{1}{2}} \tilde{\mathbf{A}} \tilde{\mathbf{D}}^{-\frac{1}{2}} \mathbf{H}^{(l)} \mathbf{W}^{(l)}\right)$$

where:
- $\tilde{\mathbf{A}} = \mathbf{A} + \mathbf{I}_N$ adds **self-loops** so each node includes its own previous state
- $\tilde{\mathbf{D}}$ is the degree matrix of $\tilde{\mathbf{A}}$: $\tilde{D}_{ii} = \sum_j \tilde{A}_{ij}$
- $\tilde{\mathbf{D}}^{-\frac{1}{2}} \tilde{\mathbf{A}} \tilde{\mathbf{D}}^{-\frac{1}{2}}$ is the **symmetric normalized adjacency matrix**
- $\mathbf{W}^{(l)}$ is a learnable linear transformation matrix
- $\sigma(\cdot)$ is a non-linear activation function (e.g. $\mathrm{ReLU}$)

<small>📖 *Source: [Bishop, C. M., & Bishop, H. (2024). Deep Learning: Foundations and Concepts. Springer (Section 13.4)](https://doi.org/10.1007/978-3-031-45468-4)*</small>

Symmetric normalization divides by $\sqrt{\tilde{D}_{ii} \tilde{D}_{jj}}$, preventing high-degree hub nodes from causing exploding feature scales.

## Graph Attention Networks (GAT)
## Dynamic Learned Edge Weighting

In GCNs, edge weights are fixed by graph degrees.
**Graph Attention Networks (GAT)** (Veličković et al. 2018) apply self-attention to learn dynamic attention weights $\alpha_{uv}$ between neighbors:

$$\alpha_{uv} = \frac{\exp\left(\mathrm{LeakyReLU}\left(\mathbf{a}^\mathrm{T} [\mathbf{W}\mathbf{h}_u \parallel \mathbf{W}\mathbf{h}_v]\right)\right)}{\sum_{k \in \mathcal{N}(v)} \exp\left(\mathrm{LeakyReLU}\left(\mathbf{a}^\mathrm{T} [\mathbf{W}\mathbf{h}_k \parallel \mathbf{W}\mathbf{h}_v]\right)\right)}$$

The node update aggregates neighbors weighted by $\alpha_{uv}$:
$$\mathbf{h}_v^{(l+1)} = \sigma\left(\sum_{u \in \mathcal{N}(v)} \alpha_{uv} \mathbf{W} \mathbf{h}_u\right)$$

Multi-head attention can be applied to stabilize learning and attend to distinct relational sub-patterns.

In [ ]:
# Node Classification Architecture: Zachary's Karate Club in PyG
if PYG_AVAILABLE:
    dataset = KarateClub()
    data = dataset[0]
    
    class KarateGCN(nn.Module):
        def __init__(self, in_dim, hidden_dim, out_dim):
            super().__init__()
            self.conv1 = GCNConv(in_dim, hidden_dim)
            self.conv2 = GCNConv(hidden_dim, out_dim)

        def forward(self, x, edge_index):
            h = F.relu(self.conv1(x, edge_index))
            out = self.conv2(h, edge_index)
            return out, h

    model_gcn = KarateGCN(in_dim=dataset.num_features, hidden_dim=4, out_dim=dataset.num_classes)
    print(f"Instantiated KarateGCN model for {data.num_nodes} nodes and {dataset.num_classes} classes.")
else:
    print("PyTorch Geometric not installed; using NetworkX fallback.")

In [ ]:
# Semi-Supervised Training Loop & Node Classification Accuracy
if PYG_AVAILABLE:
    opt = torch.optim.Adam(model_gcn.parameters(), lr=0.01)
    crit = nn.CrossEntropyLoss()

    for epoch in range(100):
        opt.zero_grad()
        out, _ = model_gcn(data.x, data.edge_index)
        loss = crit(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        opt.step()

    out, embeddings = model_gcn(data.x, data.edge_index)
    preds = out.argmax(dim=-1)
    acc = (preds == data.y).float().mean()
    print(f"Karate Club GCN Final Loss: {loss.item():.4f}")
    print(f"Full Graph Node Classification Accuracy: {acc*100:.1f}%")

In [ ]:
# Visualization helper for learned graph node embeddings
def plot_karate_embeddings():
    if not PYG_AVAILABLE:
        print("PyTorch Geometric required for live plot.")
        return
        
    with torch.no_grad():
        _, emb = model_gcn(data.x, data.edge_index)
        emb_2d = emb[:, :2].numpy()
        
    plt.figure(figsize=(6, 4.5))
    plt.scatter(emb_2d[:, 0], emb_2d[:, 1], c=data.y.numpy(), cmap='tab10', s=120, edgecolors='k', lw=1.5)
    for i in range(len(emb_2d)):
        plt.annotate(str(i), (emb_2d[i, 0]+0.05, emb_2d[i, 1]+0.05), fontsize=9)
    plt.title("Learned 2D Node Embedding Space (Karate Club)", fontsize=11)
    plt.xlabel("Latent Dim 1")
    plt.ylabel("Latent Dim 2")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
# Interactive display of learned node representations
plot_karate_embeddings()

# Summary: Graph Neural Networks & Geometric Deep Learning

1. **Non-Euclidean Relational Data**:
   - Graphs represent entities and connections via feature matrix $\mathbf{X}$ and adjacency $\mathbf{A}$.
   - Architectures must enforce permutation equivariance for node tasks and permutation invariance for graph readouts.

2. **The Message Passing Engine**:
   - MPNNs decompose computation into Message, Aggregation, and Update passes across local neighborhoods $\mathcal{N}(v)$.

3. **GCN and GAT**:
   - GCN normalizes messages symmetrically: $\tilde{\mathbf{D}}^{-\frac{1}{2}} \tilde{\mathbf{A}} \tilde{\mathbf{D}}^{-\frac{1}{2}} \mathbf{H} \mathbf{W}$.
   - GAT learns dynamic directional attention weights $\alpha_{uv}$, enabling adaptive relational weighting.